# Polymarket Edge Scanner — Research Notebook

Scratch space for exploring the public Polymarket API, sanity-checking the
fair-probability model, and reviewing recorded paper trades.

Run from the project root with the venv active (`source .venv/bin/activate`).

> Reminder: this project is research + paper trading only. No live orders.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from backend.services.polymarket_client import PolymarketClient

with PolymarketClient() as client:
    markets = client.fetch_active_markets(limit=5, min_liquidity=10000)
print(f'fetched {len(markets)} markets')
markets[0]['question'] if markets else 'no markets'

In [ ]:
from backend.services.market_collector import parse_market
from backend.services.fair_probability import MarketAsPriorModel, MarketView

model = MarketAsPriorModel()
for raw in markets:
    pm = parse_market(raw)
    if not pm:
        continue
    f = pm.fields
    bb, ba = f['best_bid'], f['best_ask']
    mid = (bb + ba) / 2 if bb is not None and ba is not None else (f['outcome_prices'] or [0.5])[0]
    view = MarketView(slug=f['slug'], mid_yes=mid, best_bid=bb, best_ask=ba,
                      spread=f['spread'], liquidity=f['liquidity'], volume_24h=f['volume_24h'],
                      one_day_price_change=f['one_day_price_change'])
    res = model.fair_probability(view)
    print(f"{f['question'][:50]:50s}  mid={mid:.3f}  fair={res.fair_prob_yes:.3f}")

In [ ]:
# Inspect recorded paper trades / metrics from the local SQLite DB.
from backend.db import session_scope
from backend.services.backtester import compute_metrics

with session_scope() as s:
    m = compute_metrics(s)
m.model_dump()